In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # Specify the GPU you want to use
from transformers import AutoTokenizer, LlamaForCausalLM
import torch
# from peft import PeftModel, PeftConfig

In [2]:
import torch

if torch.cuda.is_available():
    gpu_idx = 1  # or change to the index of your GPU
    gpu_properties = torch.cuda.get_device_properties(gpu_idx)
    total_memory_MB = gpu_properties.total_memory / (1024 ** 2)  # Convert from bytes to MB
    print(f"GPU {gpu_idx} memory: {total_memory_MB:.2f} MB")
else:
    print("CUDA not available.")


GPU 1 memory: 7816.25 MB


In [3]:
HOME_PATH = os.path.dirname(os.getcwd())
model_path = os.path.join(HOME_PATH, "saved_models", "LlamaFinetuneTestMalky","LlamaFinetune","final_model")

In [4]:
model_path

'/home/home/Desktop/research/saved_models/LlamaFinetuneTestMalky/LlamaFinetune/final_model'

In [5]:
# Debug print to check
print(f"Loading tokenizer from: {model_path}")
assert os.path.exists(model_path), "Model path does not exist!"

Loading tokenizer from: /home/home/Desktop/research/saved_models/LlamaFinetuneTestMalky/LlamaFinetune/final_model


In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)

In [7]:
tokenizer.special_tokens_map

{'bos_token': '<|begin_of_text|>',
 'eos_token': '<|end_of_text|>',
 'pad_token': '<|end_of_text|>',
 'additional_special_tokens': ['<|user|>', '<|assistant|>']}

In [8]:
# original_model_path = os.path.join(HOME_PATH,"Models", 'Llama3.2-1B-Instruct-hf')

In [9]:

model = LlamaForCausalLM.from_pretrained(
            pretrained_model_name_or_path= model_path,
            quantization_config={"load_in_8bit": True},  # Pass the quantization config here
            torch_dtype=torch.float16,
            device_map="cuda:1",  # You can uncomment this if you want automatic device allocation
            ignore_mismatched_sizes=True
        )  

# 3. Resize embeddings
model.resize_token_embeddings(len(tokenizer))

/home/home/anaconda3/lib/python3.12/site-packages/transformers/quantizers/auto.py:212: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


RuntimeError: CUDA error: invalid device ordinal
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [9]:
# #  Build the input
# user_prompt = (
#     "A 40-year-old woman undergoing chemotherapy for ovarian carcinoma has "
#     "developed progressive bilateral sensorineural hearing loss. Which chemotherapy"
#     "drug is responsible for this side effect?"
# )

user_prompt = (
    "DMPA-True:a) Failure @ 0.3/100 WYb) 150 mg/3 monthly deliveredc) Weight gaind) Glucose intolerance occure) Anemia improves"
    "A. bcde"
    "B. abcd"
    "C. abce"
    "D. abde"
)
    

full_prompt = f"<|user|>{user_prompt}<|assistant|>"

# Tokenize
inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)

In [10]:
# Generate
output = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.1,
    top_p=0.9,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)
# Decode
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

In [11]:
print(generated_text)

DMPA-True:a) Failure @ 0.3/100 WYb) 150 mg/3 monthly deliveredc) Weight gaind) Glucose intolerance occure) Anemia improvesA. bcdeB. abcdC. abceD. abde
Let's think step by step.

To solve this, let's break down each option one by one.

First, option a: Failure @ 0.3/100 WYb. This seems to be a typo or a misunderstanding. It's likely meant to be "Failure @ 0.3/100 mg/day" rather than "0.3/100 WYb." If that's the case, it's a dosage issue rather than a failure. So, this option doesn't seem to be about failure.

Next, option b: 150 mg/3 monthly delivered. This option is clearly about a dosage. It's a standard dosage for a medication, so it's not about failure.

Now, option c: Weight gain. This is a common side effect of many medications, including those for weight loss. It's not about failure but rather a known side effect.

Option d: Glucose intolerance occurs. This is a known side effect of certain medications, particularly those used to treat diabetes. It's not about failure but rather 